# Retrieving data

In [1]:
from src.datasets import save_pages_to_jsonl

save_pages_to_jsonl(
    dataset_name="FineWeb Edu 2",
    num_pages=2,
    output_path="./data/fineweb_edu_2/test_pages.jsonl",
    random_seed=163
)


# Single Merge

In [1]:
import yaml

MODEL_NAME = "test3"
yaml_config = """
slices:
  - sources:
      - model: RoyJoy/llama-jan08
        layer_range: [0, 48]
      - model: RoyJoy/llama-jan16
        layer_range: [0, 48]
merge_method: slerp
base_model: RoyJoy/llama-jan08
parameters:
  t:
    - filter: self_attn
      value: [0, 0.5, 0.3, 0.7, 1]
    - filter: mlp
      value: [1, 0.5, 0.7, 0.3, 0]
    - value: 0.5
dtype: bfloat16
"""

# Save config as yaml file
with open("config.yaml", "w", encoding="utf-8") as f:
    f.write(yaml_config)

In [ ]:
# run merge
!mergekit-yaml config.yaml merge --lazy-unpickle

In [ ]:
!pip install -qU huggingface_hub
!pip install python-dotenv

from huggingface_hub import ModelCard, ModelCardData
from jinja2 import Template

username = "deepcoreCoalbiter"

template_text = """
# {{ model_name }}
"""

# Create a Jinja template object
jinja_template = Template(template_text.strip())

# Get list of models from config
data = yaml.safe_load(yaml_config)
if "models" in data:
    models = [data["models"][i]["model"] for i in range(len(data["models"])) if "parameters" in data["models"][i]]
elif "parameters" in data:
    models = [data["slices"][0]["sources"][i]["model"] for i in range(len(data["slices"][0]["sources"]))]
elif "slices" in data:
    models = [data["slices"][i]["sources"][0]["model"] for i in range(len(data["slices"]))]
else:
    raise Exception("No models or slices found in yaml config")

# Fill the template
content = jinja_template.render(
    model_name=MODEL_NAME,
)

# Save the model card
card = ModelCard(content)
card.save('merge/README.md')


In [ ]:
from huggingface_hub import HfApi

username = "deepcoreCoalbiter"

# Read token from .env file
import os
from dotenv import load_dotenv

load_dotenv()

api = HfApi(token=os.getenv("HF_TOKEN"))

api.create_repo(repo_id=f"{username}/{MODEL_NAME}", repo_type="model")
api.upload_folder(
    repo_id=f"{username}/{MODEL_NAME}",
    folder_path="merge",
)

# Evolutionary merge with custom task

In [2]:
yaml_config = """
genome:
    models:
      - model: RoyJoy/llama-jan08
      - model: RoyJoy/llama-jan16
    merge_method: slerp
    base_model: RoyJoy/llama-jan08
    layer_granularity: 8

tasks:
  - name: fine_web_edu_2_ppl
    weight: 1.0
    metric: "perplexity"
    config:
        dataset:
            path: "json"
            name: null
            data_files: "./data/fineweb_edu_2/test_pages.json"
            split: "test"
"""

# Save config as yaml file
with open("evolve_config.yaml", "w", encoding="utf-8") as f:
    f.write(yaml_config)

In [4]:
!mergekit-evolve --task-search-path lm-eval-tasks/fineweb-edu-2 --storage-path ./evolve_storage evolve_config.yaml

Resharding models:   0%|                                  | 0/2 [00:00<?, ?it/s]2025-01-23:02:35:27,948 INFO     [evolve.py:367] Using existing resharded model at ./evolve_storage/input_models/llama-jan08_2212976643
2025-01-23:02:35:27,948 INFO     [evolve.py:367] Using existing resharded model at ./evolve_storage/input_models/llama-jan16_2101625925
Resharding models: 100%|████████████████████████| 2/2 [00:00<00:00, 6278.90it/s]
2025-01-23:02:35:27,949 INFO     [evolve.py:367] Using existing resharded model at ./evolve_storage/input_models/llama-jan08_2212976643
2025-01-23 02:35:37,590	INFO worker.py:1821 -- Started a local Ray instance.
(5_w,11)-aCMA-ES (mu_w=3.4,w_1=42%) in dimension 12 (seed=647548, Thu Jan 23 02:35:39 2025)
Received 11 genotypes
(OnDiskMergeEvaluator pid=1814125) 2025-01-23:02:35:43,670 INFO     [actors.py:100] Merging model
(OnDiskMergeEvaluator pid=1814125) 2025-01-23:02:35:43,679 INFO     [merge.py:81] Planning operations
(OnDiskMergeEvaluator pid=1814125) 2025-